# AI Workforce Capacity Planning Platform
## Notebook 03 — Demand Intelligence and Forecast Dataset Engine

**Functional implementations:**  
- Implementation 09 — Enterprise Demand Intelligence Engine  
- Implementation 10 — Enterprise Forecast Dataset Framework  

**Release remediation:**  
- Implementation 28 — Enterprise Release Remediation  

**Platform release:** v3.0.0

**Primary target:** `order_line_count`  
**Supported horizons:** 1, 7, 14, 30, 60, and 90 days  
**Current selected horizon:** 14 days  

This notebook remains a thin orchestration layer.

It:

1. loads the validated Gold daily workload dataset,
2. resolves the operational demand profile,
3. generates and validates business and temporal features,
4. builds the model-ready forecast dataset,
5. creates chronological train, validation, and test splits,
6. validates forecast-dataset metadata and execution contracts,
7. publishes Implementation 09 and 10 execution summaries.

Business logic is implemented in:

- `src/demand`
- `src/forecast`

The notebook must not duplicate reusable demand or forecasting logic.

In [0]:
%run ./00_project_setup

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Remediation
#
# Notebook:
#     03_forecasting_engine
#
# Purpose:
#     Canonical runtime imports for demand-intelligence and forecast-dataset
#     orchestration.
#
# Release:
#     v3.0.0
# =============================================================================

from dataclasses import asdict

from pyspark.sql import functions as F

from src.demand.service import DemandService
from src.forecast import ForecastDatasetService

In [0]:
# =============================================================================
# Canonical Namespace Contract
# =============================================================================

assert PLATFORM_RELEASE == "v3.0.0"
assert CANONICAL_NAMESPACE == "src.*"

assert DemandService.__module__.startswith("src.demand")
assert ForecastDatasetService.__module__.startswith("src.forecast")

print("=" * 72)
print("NOTEBOOK 03 — RELEASE CONTRACT")
print("=" * 72)
print(f"Platform release:     {PLATFORM_RELEASE}")
print(f"Canonical namespace:  {CANONICAL_NAMESPACE}")
print(f"Demand service:       {DemandService.__module__}")
print(f"Forecast service:     {ForecastDatasetService.__module__}")
print("Release contract:     PASSED")
print("=" * 72)

## Section 01 — Gold Demand Dataset  

Load the canonical daily workload dataset produced by the enterprise data pipeline.

In [0]:
GOLD_DAILY_WORKLOAD_PATH = f"{GOLD_ROOT}/daily_workload"

gold_df = (
    spark.read
    .parquet(GOLD_DAILY_WORKLOAD_PATH)
)

gold_row_count = gold_df.count()

if gold_row_count == 0:
    raise RuntimeError("Gold daily workload dataset contains no records.")

print(f"Gold path: {GOLD_DAILY_WORKLOAD_PATH}")
print(f"Gold rows: {gold_row_count:,}")

display(
    gold_df
    .orderBy(F.col("order_date").asc())
    .limit(10)
)

## Section 02 — Forecast Profile

Resolve the primary operational forecasting profile.

Daily order-line demand is the primary target because warehouse productivity, staffing, and overtime capacity are measured in order lines.

In [0]:
# =============================================================================
# Section 02 — Forecast Profile
# =============================================================================

service = DemandService()

profile = service.get_profile()

assert (
    ACTIVE_FORECAST_HORIZON_DAYS
    in profile.horizons
), (
    "Active forecast horizon is not supported by the "
    f"demand profile. Active={ACTIVE_FORECAST_HORIZON_DAYS}; "
    f"Supported={profile.horizons}"
)

forecast_service = ForecastDatasetService(
    spark=spark,
    target_column=profile.target,
    forecast_horizon=ACTIVE_FORECAST_HORIZON_DAYS,
)


profile_display_df = spark.createDataFrame(
    [
        {
            "profile_name": profile.name,
            "target_column": profile.target,
            "forecast_horizons": ", ".join(
                str(horizon)
                for horizon in profile.horizons
            ),
            "selected_dataset_horizon": (
                forecast_service.forecast_horizon
            ),
            "business_feature_count": len(
                profile.business_features
            ),
            "ml_feature_groups": ", ".join(
                profile.ml_features
            ),
        }
    ]
)

display(profile_display_df)

## Section 03 — Demand Dataset Summary

Validate timeline continuity, date uniqueness, target completeness, and approved business sanity rules.

In [0]:
summary = service.summarize_dataset(
    dataframe=gold_df
)

summary_df = spark.createDataFrame(
    [asdict(summary)]
)

display(summary_df)

if not summary.validation_passed:
    raise RuntimeError(
        "Gold demand dataset failed Demand Intelligence validation."
    )

## Section 04 — Forecast Feature Generation

Generate:

- operational business features,
- historical lag features,
- leakage-safe rolling statistics,
- historical trend and growth features,
- calendar seasonality indicators.

The current day's target is excluded from all predictive temporal features.

In [0]:
forecast_df = service.build_forecast_dataset(
    dataframe=gold_df,
    validate_input=True,
    validate_output=False,
)

forecast_row_count = forecast_df.count()
forecast_column_count = len(forecast_df.columns)

print(f"Forecast rows: {forecast_row_count:,}")
print(f"Forecast columns: {forecast_column_count:,}")

## Section 05 — Forecast Dataset Validation

Validate the generated business, lag, rolling, trend, and seasonality feature contracts.

Null values in initial historical warm-up rows are expected and are not treated as defects.

In [0]:
service.validate_forecast_dataset(
    dataframe=forecast_df,
    profile_name=profile.name,
)

print("Demand Intelligence Engine validation passed.")

## Section 06 — Feature Inspection

Inspect representative business and leakage-safe temporal features in chronological order.

In [0]:
selected_columns = [
    "order_date",
    "order_line_count",
    "avg_lines_per_order",
    "avg_units_per_line",
    "order_line_count_lag_1",
    "order_line_count_lag_7",
    "order_line_count_rolling_mean_7",
    "order_line_count_rolling_mean_30",
    "order_line_count_change_7",
    "order_line_count_growth_rate_7",
    "order_line_count_momentum_7_30",
    "day_of_month",
    "quarter",
    "is_month_start",
    "is_month_end",
]

display(
    forecast_df
    .select(*selected_columns)
    .orderBy(F.col("order_date").asc())
)

## Section 07 — Execution Summary

Publish the final operational status of the Demand Intelligence Engine.

In [0]:
business_feature_count = len(profile.business_features)

ml_feature_count = (
    forecast_column_count
    - len(gold_df.columns)
    - business_feature_count
)

In [0]:
execution_summary = [
    {
        "implementation": "09",
        "release_remediation": "28",
        "platform_release": PLATFORM_RELEASE,
        "canonical_namespace": CANONICAL_NAMESPACE,
        "engine": "Demand Intelligence Engine",
        "profile_name": profile.name,
        "target_column": profile.target,

        "business_features_generated": business_feature_count,
        "ml_features_generated": ml_feature_count,
        "total_dataset_columns": forecast_column_count,

        "source_path": GOLD_DAILY_WORKLOAD_PATH,
        "source_rows": gold_row_count,
        "forecast_rows": forecast_row_count,

        "start_date": summary.start_date,
        "end_date": summary.end_date,
        "missing_dates": summary.missing_dates,
        "duplicate_dates": summary.duplicate_dates,
        "validation_passed": summary.validation_passed,

        "status": "COMPLETED",
    }
]

display(
    spark.createDataFrame(execution_summary)
)

## Section 08 — Enterprise Forecast Dataset

Remove incomplete historical warm-up rows and create leakage-safe temporal training, validation, and test datasets.

In [0]:
forecast_bundle = forecast_service.build(
    dataframe=forecast_df,
    persist=False,
)

print("Enterprise Forecast Dataset build passed.")

In [0]:
forecast_dataset_summary_df = spark.createDataFrame(
    [forecast_bundle.as_summary_dict()]
)

display(forecast_dataset_summary_df)

## Section 09 — Temporal Split Boundaries

Inspect the chronological boundaries and row counts of the training, validation, and test datasets.

In [0]:
split_summary = [
    {
        "split": forecast_bundle.train.name,
        "rows": forecast_bundle.train.row_count,
        "start_date": forecast_bundle.train.start_date,
        "end_date": forecast_bundle.train.end_date,
    },
    {
        "split": forecast_bundle.validation.name,
        "rows": forecast_bundle.validation.row_count,
        "start_date": forecast_bundle.validation.start_date,
        "end_date": forecast_bundle.validation.end_date,
    },
    {
        "split": forecast_bundle.test.name,
        "rows": forecast_bundle.test.row_count,
        "start_date": forecast_bundle.test.start_date,
        "end_date": forecast_bundle.test.end_date,
    },
]

display(spark.createDataFrame(split_summary))


## Section 10 — Forecast Dataset Bundle Summary

Publish the complete metadata describing the enterprise-ready forecasting dataset generated by the Forecast Dataset Service.

In [0]:
bundle_summary = {
    "dataset_name": forecast_bundle.metadata.dataset_name,
    "dataset_version": forecast_bundle.metadata.dataset_version,
    "forecast_horizon": forecast_bundle.metadata.forecast_horizon,
    "target_column": forecast_bundle.metadata.target_column,
    "date_column": forecast_bundle.metadata.date_column,
    "split_strategy": forecast_bundle.metadata.split_strategy,
    "source_rows": forecast_bundle.metadata.source_rows,
    "warmup_rows_removed": (
        forecast_bundle.metadata.warmup_rows_removed
    ),
    "model_ready_rows": forecast_bundle.metadata.model_ready_rows,
    "train_rows": forecast_bundle.train.row_count,
    "validation_rows": forecast_bundle.validation.row_count,
    "test_rows": forecast_bundle.test.row_count,
    "total_columns": forecast_bundle.metadata.total_columns,
    "validation_passed": (
        forecast_bundle.summary.validation_passed
    ),
    "status": forecast_bundle.summary.status,
}

display(
    spark.createDataFrame([bundle_summary])
)